# Demo 20 — Several input files: one file per subject

Lab data often come as one file per subject or session, named after it:
`subject_308.csv`, `subject_309.csv`, ... with no subject column inside, because
the file name says whose data it is. `options.in_file` can name a whole folder
(or a pattern such as `'data/subject_*.csv'`, or files separated by `;`), and
kbstatpy stacks the files row by row. Each row's source goes into a new column
`file`, the file name without extension, and that column is the subject ID.

The data are those of demo 6, `sleepstudy` (lme4; Belenky et al. 2003): reaction
times of 18 subjects over ten days of sleep restriction, here a rested block
(days 0–4) against a deprived one (days 5–9), stored as 18 files of ten rows each.

## Run on Google Colab

On [Google Colab](https://colab.research.google.com)? Run the cell below first —
it installs kbstatpy, its R packages, and the demo data (~3–5 min the first time).
It is a no-op when you run this notebook locally from the kbstatpy source tree.
Then run the cells below to see the tables and figures rendered inline.

In [ ]:
# Google Colab only: install kbstatpy + its R packages + the demo data.
# (Does nothing when the notebook runs locally from the source tree.)
import sys
if 'google.colab' in sys.modules:
    !curl -sSL https://raw.githubusercontent.com/kimbostroem/kbstatpy/master/demos/colab_setup.sh | bash

## Setup

In [ ]:
import os

import pandas as pd
from kbstatpy import Kbstat, KbstatOptions

## The files

One file per subject; inside, no subject column.

In [ ]:
folder = os.path.join(KbstatOptions().demo_dir, 'data/sleepstudy_subjects')
print(sorted(os.listdir(folder)))
pd.read_csv(os.path.join(folder, 'subject_308.csv'))

## Read the folder and fit

`in_file` names the folder, so every data file in it is read, in natural order
(`subject_2` would come before `subject_10`), the same on every machine. The model
is demo 6's, with the source file as subject:
`Reaction ~ Period + (1 + Period|file)`. The results are identical to demo 6, which
reads the same data from one file with a Subject column. `rename = 'file -> Subject'`
shows the column as Subject in the tables and the plot. `out_dir` is empty, so
results render inline only.

In [ ]:
options = KbstatOptions()
options.in_file  = os.path.join(options.demo_dir, 'data/sleepstudy_subjects')  # a folder: all its data files
options.out_dir  = ''                 # inline only; set a folder to also save
options.y        = 'Reaction'
options.y_units  = 'ms'
options.x        = 'Period'
options.id       = 'file'             # the source file of each row is the subject
options.slope    = 'Period'           # each subject gets their own Period effect
options.rename   = 'Reaction -> ReactionTime; Period -> Day; file -> Subject'

kb = Kbstat(options)
kb.run();

## The stacked data

What the model was fitted to: 180 rows, the source of each in the column `file`.

In [ ]:
kb.output.results[0].data.head(12)

## Interpretation

- Reading a folder gives exactly the analysis of the single stacked file (demo 6):
  the same ANOVA, variance components and post-hoc comparison.
- The `file` column is an ordinary column: besides `id`, it can be a factor, a
  filter in `constraints` (`file != "subject_308"`), or `split = 'file'` for one
  separate analysis per file.
- Summary.txt lists every file read with its row count, so a results folder
  records which data went in. Files that lack a column of the others are named in
  a warning, since their rows would otherwise leave the fit unnoticed.